# Suite SVA — The SystemVerilog Assertions dialect

SystemVerilog Assertions' sequences and properties (IEEE 1800 16.7 to 16.12) are a dialect of the SystemVerilog family,
`Expressions.SystemVerilog.Assertions`: mbse-programs' kinds, in its shapes, each a branch of its `Term`, which lists
every branch of `Expressions`' `Term` besides its own, since a position that holds a property holds a sequence, a
property or an expression. A call's arguments hold it, so a tree mixes its kinds with the family's; it is validated and
printed from any of the family's holders, its editions are data, and evaluation refuses each of its kinds by name.

In [ ]:
import { Framework } from "@mbse/expressions";
import * as IEEE1164 from "@mbse/expressions/IEEE1164/Expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as Unicode from "@mbse/expressions/Unicode/Expressions";
import * as Patterns from "@mbse/expressions/Patterns/Expressions";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as SVAssertions from "@mbse/expressions/SystemVerilog/Assertions";
import * as SVConstraints from "@mbse/expressions/SystemVerilog/Constraints";
import * as SVPatterns from "@mbse/expressions/SystemVerilog/Patterns";
import * as SVPreprocessor from "@mbse/expressions/SystemVerilog/Preprocessor";
import * as SVTypes from "@mbse/expressions/SystemVerilog/Types";
import { Domains as D, Standards as FS, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors, JSON, Validators, YAML } from "@mbse/schemas/Framework";
import { assert, equal, map, raises, same } from "./support.js";

const { NotImplementedError, ValueError } = Errors;
const PE = Patterns.Expressions;
const [SVA, SVAText] = [SVAssertions.Expressions, SVAssertions.Text];
const [SV, SVV, SVText] = [SystemVerilog.Expressions, SystemVerilog.Evaluators, SystemVerilog.Text];
const SVT = SVTypes.Expressions;

const store: any = new Builders.Store(); // the SystemVerilog family's six dialects, after the standard dialects they use
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, IEEE1164.install, Unicode.install,
  SystemVerilog.install, SVTypes.install, SVPatterns.install, SVAssertions.install, SVConstraints.install,
  SVPreprocessor.install]) {
  store.transaction(install);
}
for (const define of [SystemVerilog.define, SVAssertions.define]) { // SystemVerilog's types, and the editions Assertions covers
  store.transaction(define);
}
const Term: any = store.registered("Expressions.Framework.Term");
const [UNION, EXPRESSIONS]: any[] = [store.registered(SVA.DIALECT.union), store.registered(SV.DIALECT.union)];
const KINDS = new Map<string, any>(UNION.branches.map((b: any) => [b.name, b.schema])); // its Term's branches' schemas, by tag
const BOUND = ["a", "b", "c", "d", "clk", "rst", "en", "n", "x", "f"]; // the names its samples refer to

/** The term value `spec` writes, the root of a holder of the family's dialect `holder` written in a transaction. */
function held(spec: unknown, holder = "Assertions"): any {
  return F.term(store.transaction(() => store.Expressions.SystemVerilog[holder].Expression((e: any) => e.root(spec))));
}

/** The problems of the term `spec` writes, held by Assertions' holder, with `bound` bound. */
function validate(spec: unknown, bound: string[] = BOUND): string[] {
  return SVA.DIALECT.validate(held(spec), bound);
}

/** The term `spec` writes, held by Assertions' holder, as SystemVerilog source. */
function text(spec: unknown): string {
  return SVText.ToText(held(spec));
}

/** Whether a term of its Term reads back the same from JSON and from YAML. */
function roundTrips(value: unknown): boolean {
  return ([[JSON.ToJSON(store), JSON.FromJSON(store)], [YAML.ToYAML(store), YAML.FromYAML(store)]] as [any, any][]).every(
    ([encode, decode]) => F.same(decode(UNION, encode(UNION, value)), value));
}

/** A set's names, sorted. */
const names = (set: ReadonlySet<string>): string[] => [...set].sort();
const spelled = (text: string) => (i: any) => i.spelling(text); // an identifier, of its spelling
const name = (text: string) => (u: any) => u.nameexpression((n: any) => n.name((i: any) => i.identifier(spelled(text))));
const [a, b, c, d, clk, rst, en, n, x] = ["a", "b", "c", "d", "clk", "rst", "en", "n", "x"].map(name);
const [one, two, three] = [SV.literal(1n), SV.literal(2n), SV.literal(3n)];
const dollar = (u: any) => u.dollarexpression((r: any) => r); // `$`
const cycles = (low: unknown, high: unknown) => (u: any) => u.cyclerange((r: any) => r.low(low).high(high)); // `low:high`
const event = (edge: string, signal: unknown) => (e: any) => e.edge(edge).expression(signal); // `edge signal`
const posedge = (k: any) => k.events([event("posedge", clk)]); // a clock, `@(posedge clk)`
const seq = (left: unknown, op: string, right: unknown) => (u: any) => u.binarysequence((s: any) => s.left(left).operator(op).right(right));
const prop = (left: unknown, op: string, right: unknown) => (u: any) => u.binaryproperty((s: any) => s.left(left).operator(op).right(right));
const implies = (ante: unknown, op: string, cons: unknown) => (u: any) => u.implicationproperty(
  (s: any) => s.antecedent(ante).operator(op).consequent(cons));
const unary = (op: string, operand: unknown, range: unknown = null) => (u: any) => u.unaryproperty(
  (s: any) => (range !== null ? s.operator(op).range(range) : s.operator(op)).operand(operand));
const delay = (first: unknown, ...steps: [unknown, unknown][]) => (u: any) => u.delaysequence((s: any) => (first !== null ? s.first(first) : s).steps(
  steps.map((step) => (t: any) => t.delay(step[0]).sequence(step[1]))));
const repeat = (sequence: unknown, op: string, count: unknown) => (u: any) => u.repetitionsequence(
  (s: any) => s.sequence(sequence).operator(op).count(count));

<a id="SVA-01"></a>
## SVA-01 · Its union and holder: its kinds after `Expressions`' branches, in mbse-programs' shapes, the positions that hold its `Term`, its holder validated, linked by `HasRoot` and round-tripped

In [ ]:
// Its kinds: mbse-programs' sequences and properties, a property spec and a cycle range, after Expressions' branches and
// invalid syntax; the Preprocessor's macro usage after them, and the Constraints' distexpression once its install adds it.
{
  assert(same(SVA.TAGS, [...SVA.SEQUENCES, ...SVA.PROPERTIES, "propertyspec", "cyclerange"]) && SVA.TAGS.length === 17);
  assert(same(SVA.SEQUENCES, ["delaysequence", "repetitionsequence", "binarysequence", "parenthesizedsequence",
    "firstmatchsequence", "clockedsequence"]));
  assert(same(SVA.PROPERTIES, ["implicationproperty", "binaryproperty", "unaryproperty", "strengthproperty", "abortproperty",
    "conditionalproperty", "caseproperty", "parenthesizedproperty", "clockedproperty"]));
  assert(same(UNION.branches.map((b: any) => b.name).filter((tag: string) => tag !== "distexpression"),
    [...SV.TAGS, "error", "missing", ...SVA.TAGS, "macrousage"]));
  assert(same([...SVA.DIALECT.kinds().keys()], [...SVA.TAGS, "error", "missing"]) && same(SVA.HELPERS, [
    "DelayStep", "PropertyCaseItem", "EventControl", "EventExpression"]));
  assert(SVA.DIALECT.name() === "SystemVerilog Assertions" && SVA.DIALECT.tag === "systemverilog_assertions");
  assert(Term.branches.find((b: any) => b.name === "systemverilog_assertions").schema === UNION); // the framework's Term's branch
  const expressions = new Map<string, any>(EXPRESSIONS.branches.map((b: any) => [b.name, b.schema]));
  assert(SV.TAGS.every((tag) => KINDS.get(tag) === expressions.get(tag))); // one schema each
  // Each kind a value object, its properties in mbse-programs' order; every operand of a sequence or a property, and what
  // the standard writes `expression_or_dist`, holds its Term; a match item, a cycle range's bound and an event Expressions'.
  const PROPERTIES: Record<string, string[]> = {
    delaysequence: ["first", "steps"], repetitionsequence: ["sequence", "operator", "count"],
    binarysequence: ["left", "operator", "right"], parenthesizedsequence: ["sequence", "items"],
    firstmatchsequence: ["sequence", "items"], clockedsequence: ["clock", "sequence"],
    implicationproperty: ["antecedent", "operator", "consequent"], binaryproperty: ["left", "operator", "right"],
    unaryproperty: ["operator", "range", "operand"], strengthproperty: ["keyword", "sequence"],
    abortproperty: ["keyword", "condition", "operand"], conditionalproperty: ["condition", "consequence", "alternative"],
    caseproperty: ["expression", "items"], parenthesizedproperty: ["property"], clockedproperty: ["clock", "property"],
    propertyspec: ["clock", "disable", "property"], cyclerange: ["low", "high"],
  };
  assert(SVA.TAGS.every((tag) => same([...KINDS.get(tag).properties.keys()], PROPERTIES[tag]!) && !KINDS.get(tag).ref));
  assert(SVA.TAGS.every((tag) => { const own = KINDS.get(tag).name.split(".").pop();
    return KINDS.get(tag).name === `Expressions.SystemVerilog.Assertions.${own}` && own.toLowerCase() === tag; }));
  const helper = (name: string): any => store.registered(`Expressions.SystemVerilog.Assertions.${name}`);
  assert(equal(SVA.HELPERS.map((h) => [...helper(h).properties.keys()]), [
    ["delay", "sequence"], ["expressions", "body"], ["events"], ["edge", "expression", "condition"]]));
  for (const [tag, slot] of [["binarysequence", "left"], ["delaysequence", "first"], ["unaryproperty", "range"],
    ["conditionalproperty", "condition"], ["abortproperty", "condition"], ["caseproperty", "expression"],
    ["propertyspec", "disable"], ["propertyspec", "property"]] as [string, string][]) {
    assert(KINDS.get(tag).properties.get(slot).schema === UNION, `${tag} ${slot}`);
  }
  assert(KINDS.get("cyclerange").properties.get("low").schema === EXPRESSIONS && KINDS.get("parenthesizedsequence").properties.get(
    "items").schema.element === EXPRESSIONS && helper("EventExpression").properties.get("condition").schema === EXPRESSIONS);
  assert(helper("DelayStep").properties.get("delay").schema === UNION && helper("PropertyCaseItem").properties.get(
    "expressions").schema.element === UNION && KINDS.get("clockedsequence").properties.get("clock").schema === helper("EventControl"));
  // How the framework reads them: a delay's steps, pairs of terms, its variadic; a clock and a case's items helpers.
  assert(equal(F._parts(KINDS.get("delaysequence")), [["first"], "steps", []]) && equal(F._parts(KINDS.get("parenthesizedsequence")), [
    ["sequence"], "items", []]));
  assert(equal(F._parts(KINDS.get("clockedsequence")), [["sequence"], null, ["clock"]]) && equal(F._parts(KINDS.get("caseproperty")), [
    ["expression"], null, ["items"]]) && equal(F._parts(KINDS.get("propertyspec")), [["disable", "property"], null, ["clock"]]));
  assert(SVA.TAGS.every((tag) => SVA.DIALECT.kinds().get(tag)!.role === F.APPLICATION));
  // Its holder (Q2, "(a)") refines the framework's Expression, its root narrowed to its Term: it validates, holds a term of
  // its dialect, round-trips through JSON and YAML, and HasRoot links it, as any expression.
  const Expression: any = store.registered("Expressions.Framework.Expression");
  const Holder: any = store.registered("Expressions.SystemVerilog.Assertions.Expression");
  assert(Holder.ref && Holder.refines === Expression && F.isExpression(Holder) && Holder.properties.get("root").schema === UNION);
  const [spec, linked] = store.transaction(() => {
    const spec = store.Expressions.SystemVerilog.Assertions.Expression((e: any) => e.root((u: any) => u.propertyspec(
      (s: any) => s.clock(posedge).disable(rst).property(implies(a, "|=>", b)))));
    const linked = store.Schemas.OfExpression((s: any) => s.name("sva.Handshake").expression(
      (h: any) => h.insert((e: any) => e.expression(spec))));
    return [spec, linked];
  });
  assert(Validators.Validate(store)(Holder, spec).length === 0 && SVA.DIALECT.validate(spec, BOUND).length === 0);
  assert(F.dialectOf(F.term(spec)) === SVA.DIALECT.union && roundTrips(F.term(spec)));
  assert(F.same(JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, spec)).root, spec.root));
  assert(same([...(new Builders.OfExpression.Builder(linked, false, store) as any).expression].map((e: any) => F.same(e.expression, spec)), [true]));
  assert(SVText.ToText(spec) === SVAText.ToText(spec) && SVText.ToText(spec) === "@(posedge clk) disable iff (rst) a |=> b");
  raises(Error, () => store.transaction(() => store.Expressions.SystemVerilog.Assertions.Expression( // the schema refuses a kind its
    (e: any) => e.root((t: any) => t.integeratomtype((i: any) => i.keyword("int")))))); // root's union lacks: a Types term
}

<a id="SVA-02"></a>
## SVA-02 · Sequences: each kind written through the builders and validated, its checks with mbse-programs' messages, and no kind binding a name

In [ ]:
// Each sequence (16.7 to 16.10), written through the builders: validated, its names free (none of its kinds binds one),
// and its checks, with mbse-programs' choices and messages.
{
  const matchItem = SV.binary("+", x, one); // a match item, an expression
  const SEQUENCES: unknown[] = [
    delay(a, [one, b], [cycles(one, dollar), c]), // a ##1 b ##[1:$] c
    delay(null, [two, a]), // ##2 a: a sequence that starts with a delay
    repeat(a, "*", two), repeat(b, "->", cycles(one, three)), repeat(c, "=", n),
    seq(a, "and", b), seq(a, "or", b), seq(a, "intersect", b), seq(a, "within", b), seq(en, "throughout", b),
    (u: any) => u.parenthesizedsequence((s: any) => s.sequence(a).items([matchItem])),
    (u: any) => u.firstmatchsequence((s: any) => s.sequence(seq(a, "or", b))),
    (u: any) => u.clockedsequence((s: any) => s.clock(posedge).sequence(a)),
  ];
  assert(SEQUENCES.every((spec) => validate(spec).length === 0));
  assert(same(SEQUENCES.map((spec) => F.kindOf(held(spec))[0]), [
    "delaysequence", "delaysequence", ...Array(3).fill("repetitionsequence"), ...Array(5).fill("binarysequence"),
    "parenthesizedsequence", "firstmatchsequence", "clockedsequence"]));
  assert(same(names(SY.free(held(SEQUENCES[0]), SVA.DIALECT)), ["a", "b", "c"]) && same(names(SY.free(held(SEQUENCES[4]), SVA.DIALECT)), ["c", "n"]));
  assert(same(names(SY.free(held(SEQUENCES[10]), SVA.DIALECT)), ["a", "x"]) && same(names(SY.free(held(SEQUENCES[12]), SVA.DIALECT)), ["a", "clk"]));
  assert(!SVA.TAGS.some((tag) => F.isBinder(KINDS.get(tag)))); // no kind of it binds a name: no `Let` refined
  assert(same(validate(SEQUENCES[0], ["a", "b"]), ["sequence 1: nameexpression 'c' is not bound"])); // a step's members, by index
  assert(SVA.DIALECT.infer(held(SEQUENCES[5]), { a: D.Anything, b: D.Anything }) === D.Anything); // nothing it evaluates
  const cases: [unknown, string[]][] = [
    [repeat(a, "+", two), ["RepetitionSequence.operator cannot be '+'"]],
    [seq(a, "xor", b), ["BinarySequence.operator cannot be 'xor'"]],
    [(u: any) => u.binarysequence((s: any) => s.left(a).right(b)), ["a binarysequence needs an operator"]],
    [(u: any) => u.binarysequence((s: any) => s.left(a).operator("and")), ["a binarysequence needs a right"]],
    [(u: any) => u.delaysequence((s: any) => s.steps([(t: any) => t.delay(one)])), ["a delaysequence needs a sequence 0"]],
    [(u: any) => u.clockedsequence((s: any) => s.sequence(a)), ["a clockedsequence needs a clock"]],
    [(u: any) => u.clockedsequence((s: any) => s.clock((k: any) => k.events([event("rise", clk)])).sequence(a)),
      ["EventExpression.edge cannot be 'rise'"]], // a helper's choice, read within the clock
    [(u: any) => u.clockedsequence((s: any) => s.clock((k: any) => k.events([(e: any) => e.edge("posedge")])).sequence(a)),
      ["a clockedsequence needs a clock events 0 expression"]],
    [(u: any) => u.clockedsequence((s: any) => s.clock((k: any) => k.events([event("edge", name("q"))])).sequence(a)),
      ["clock events 0 expression: nameexpression 'q' is not bound"]],
    [(u: any) => u.clockedsequence((s: any) => s.clock((k: any) => k.events([(e: any) => e.expression(clk).condition(en)])).sequence(a)),
      []], // an edge and a condition may be absent
    [(u: any) => u.parenthesizedsequence((s: any) => s.sequence(a).items([name("q")])), ["argument 0: nameexpression 'q' is not bound"]],
  ];
  for (const [spec, problems] of cases) assert(same(validate(spec), problems), `${problems} ${validate(spec)}`);
}

<a id="SVA-03"></a>
## SVA-03 · Properties, a property spec and a cycle range: each kind validated, its checks, and the names it needs

In [ ]:
// Each property (16.12), a property spec and a cycle range: validated, its names free, and its checks.
{
  const caseItems = (i: unknown, j: unknown) => [(t: any) => t.expressions([one, two]).body(i), (t: any) => t.body(j)]; // `1, 2: i; default: j;`
  const PROPERTIES: unknown[] = [
    implies(a, "|->", b), implies(a, "|=>", b), implies(a, "#-#", b), implies(a, "#=#", b),
    ...["and", "or", "iff", "implies", "until", "s_until", "until_with", "s_until_with"].map((op) => prop(a, op, b)),
    ...["not", "nexttime", "s_nexttime", "always", "s_always", "eventually", "s_eventually"].map((op) => unary(op, a)),
    unary("nexttime", a, two), unary("always", a, cycles(one, three)),
    ...["strong", "weak"].map((k) => (u: any) => u.strengthproperty((s: any) => s.keyword(k).sequence(seq(a, "or", b)))),
    ...["accept_on", "reject_on", "sync_accept_on", "sync_reject_on"].map((k) => (u: any) => u.abortproperty(
      (s: any) => s.keyword(k).condition(rst).operand(a))),
    (u: any) => u.conditionalproperty((s: any) => s.condition(en).consequence(a)),
    (u: any) => u.conditionalproperty((s: any) => s.condition(en).consequence(a).alternative(b)),
    (u: any) => u.caseproperty((s: any) => s.expression(n).items(caseItems(a, b))),
    (u: any) => u.parenthesizedproperty((s: any) => s.property(implies(a, "|->", b))),
    (u: any) => u.clockedproperty((s: any) => s.clock(posedge).property(a)),
    (u: any) => u.propertyspec((s: any) => s.clock(posedge).disable(rst).property(a)),
    (u: any) => u.propertyspec((s: any) => s.property(a)), // with neither clock nor reset
    cycles(one, dollar),
  ];
  assert(PROPERTIES.every((spec) => validate(spec).length === 0));
  assert(same([...new Set(PROPERTIES.map((spec) => F.kindOf(held(spec))[0]))].sort(), [...SVA.PROPERTIES, "propertyspec", "cyclerange"].sort()));
  assert(same(names(SY.free(held(PROPERTIES[PROPERTIES.length - 6]), SVA.DIALECT)), ["a", "b", "n"])
    && same(names(SY.free(held(PROPERTIES[PROPERTIES.length - 3]), SVA.DIALECT)), ["a", "clk", "rst"]));
  const cases: [unknown, string[]][] = [
    [implies(a, "->", b), ["ImplicationProperty.operator cannot be '->'"]],
    [prop(a, "xor", b), ["BinaryProperty.operator cannot be 'xor'"]],
    [unary("never", a), ["UnaryProperty.operator cannot be 'never'"]],
    [(u: any) => u.unaryproperty((s: any) => s.operand(a)), ["an unaryproperty needs an operator"]],
    [(u: any) => u.strengthproperty((s: any) => s.keyword("firm").sequence(a)), ["StrengthProperty.keyword cannot be 'firm'"]],
    [(u: any) => u.abortproperty((s: any) => s.keyword("abort_on").condition(rst).operand(a)),
      ["AbortProperty.keyword cannot be 'abort_on'"]],
    [(u: any) => u.abortproperty((s: any) => s.condition(rst).operand(a)), ["an abortproperty needs a keyword"]],
    [(u: any) => u.conditionalproperty((s: any) => s.consequence(a)), ["a conditionalproperty needs a condition"]],
    [(u: any) => u.caseproperty((s: any) => s.expression(n).items([(t: any) => t.expressions([one])])),
      ["a caseproperty needs an items 0 body"]],
    [(u: any) => u.caseproperty((s: any) => s.expression(n).items([(t: any) => t.expressions([name("q")]).body(a)])),
      ["items 0 argument 0: nameexpression 'q' is not bound"]],
    [(u: any) => u.clockedproperty((s: any) => s.property(a)), ["a clockedproperty needs a clock"]],
    [(u: any) => u.propertyspec((s: any) => s.disable(name("q"))), ["disable: nameexpression 'q' is not bound",
      "a propertyspec needs a property"]],
    [unary("always", a, (u: any) => u.cyclerange((r: any) => r.low(one))), ["range: a cyclerange needs a high"]],
  ];
  for (const [spec, problems] of cases) assert(same(validate(spec), problems), `${problems} ${validate(spec)}`);
}

<a id="SVA-04"></a>
## SVA-04 · A tree mixing its kinds with the family's, validated and printed from its holder and from `Expressions`', read by each dialect of the family and refused by another

In [ ]:
// A tree mixing its kinds with the family's: an expression, a cast to a Types term and a macro usage as a sequence's
// operands, validated and printed from its holder; and a sequence as a call's argument, which holds its Term, validated
// and printed from Expressions' holder, read through every dialect of the family; another dialect refuses it.
{
  const int = (t: any) => t.integeratomtype((i: any) => i.keyword("int"));
  const macro = (u: any) => u.macrousage((m: any) => m.name(spelled("READY")));
  const mixed = (u: any) => u.propertyspec((s: any) => s.clock((k: any) => k.events([event("posedge", clk), (e: any) => e.edge("negedge").expression(
    rst).condition(en)])).disable(SV.binary("||", rst, SV.unary("!", en))).property(implies(
    delay(SV.binary("==", SV.cast(int, a), one), [n, macro]), "|=>", unary("always", seq(b, "or", SV.select(c, d))))));
  const spec = held(mixed);
  assert(SVA.DIALECT.validate(spec, BOUND).length === 0 && same(names(SY.free(spec, SVA.DIALECT)), ["a", "b", "c", "clk", "d", "en", "n", "rst"]));
  assert(same(SVA.DIALECT.validate(spec, BOUND.filter((bound) => bound !== "d")), [
    "property: consequent: operand: right: index: nameexpression 'd' is not bound"]));
  assert(SVText.ToText(spec) === "@(posedge clk or negedge rst iff en) disable iff (rst || !en) "
    + "int'(a) == 1 ##n `READY |=> always b or c[d]");
  const cast = F.form(F.form(F.form(F.form(F.form(F.term(spec)).arguments[1]).arguments[0]).arguments[0]).arguments[0]).arguments[0];
  assert(F.dialectOf(cast) === SVT.DIALECT.union && SVT.DIALECT.validate(cast).length === 0); // a Types term, within Expressions'
  const called = held(SV.call("f", delay(a, [one, b]), x), "Expressions"); // f(a ##1 b, x): a sequence is an argument
  assert(SV.DIALECT.validate(called, BOUND).length === 0 && SVText.ToText(called) === "f(a ##1 b, x)");
  const argument = F.form(called).arguments[1];
  assert(F.dialectOf(argument) === SVA.DIALECT.union && F.kindOf(argument)[0] === "delaysequence");
  assert([SV.DIALECT, ...SV.DIALECT.reads()].every((dialect) => dialect.kind(argument)[2] === SVA.DIALECT.kinds().get("delaysequence")));
  assert(same(SV.DIALECT.validate(held(SV.call("f", delay(a, [one, name("q")])), "Expressions"), BOUND), [
    "argument 0: sequence 0: nameexpression 'q' is not bound"]));
  assert(same(PE.DIALECT.validate(argument), ["a SystemVerilog Assertions delaysequence is not a Patterns term"]));
  raises(ValueError, () => PE.DIALECT.kind(argument), "a SystemVerilog Assertions delaysequence is not a Patterns term");
}

<a id="SVA-05"></a>
## SVA-05 · Invalid syntax wherever its `Term` stands: reported, printed as written, and refused by evaluation

In [ ]:
// Invalid syntax (the framework's `error` and `missing`) stands wherever its Term does: as a root, an operand, a delay or
// a case item's body; reported as a problem, printed as written (a missing term as nothing), and refused by evaluation.
{
  const error = (written: string) => (u: any) => u.error((e: any) => e.text(written));
  const missing = (kind: string) => (u: any) => u.missing((m: any) => m.kind(kind));
  assert(same(validate(error("##")), ["invalid syntax '##'"]) && same(validate(missing("Sequence")), ["missing Sequence"]));
  assert(same(validate(seq(error("@@"), "and", missing("Sequence"))), ["left: invalid syntax '@@'", "right: missing Sequence"]));
  assert(same(validate(delay(a, [error("#1"), b])), ["delay 0: invalid syntax '#1'"]));
  assert(same(validate((u: any) => u.caseproperty((s: any) => s.expression(n).items([(t: any) => t.body(missing("Property"))]))), [
    "items 0 body: missing Property"]));
  assert(text(seq(error("@@"), "and", missing("Sequence"))) === "@@ and ");
  assert(text(delay(a, [error("#1"), b])) === "a ##(#1) b" && text(unary("always", error("p q"))) === "always p q");
  assert(text(implies(missing("Sequence"), "|->", b)) === " |-> b");
  raises(ValueError, () => SVV.OfAny(held(error("##")), {}), "cannot evaluate an invalid expression: invalid syntax '##'");
}

<a id="SVA-06"></a>
## SVA-06 · Its editions as data: IEEE Std 1800-2005 to 1800-2023, each with its own kinds it has

In [ ]:
// Its editions are data (CAP-268), written by its define: IEEE 1800's five, each owned by its designation, with its formal
// reference's parts and its common name, and each with its own kinds it has, from mbse-programs' SINCE: a strength, an
// abort and a case property arrived in IEEE 1800-2009, every other kind in IEEE 1800-2005. Invalid syntax, the kinds of
// the dialects it reads and Verilog's editions (IEEE 1364) are none of its.
{
  const IEEE1800 = [2005, 2009, 2012, 2017, 2023].map((year) => `IEEE Std 1800-${year}`);
  const LABELS = [2005, 2009, 2012, 2017, 2023].map((year) => `SystemVerilog-${year}`);
  const singleton = store.singleton("Expressions.SystemVerilog.Assertions");
  assert(same([...FS.standards(singleton).keys()], IEEE1800) && equal(SVA.SINCE, new Map([["strengthproperty", 4], ["abortproperty", 4],
    ["caseproperty", 4]])));
  assert(same(FS.tags(FS.find(singleton, "SystemVerilog-2009")!), ["IEEE Std 1800-2009", "SystemVerilog-2009"]));
  assert(FS.find(singleton, "IEEE Std 1800-2009") === FS.find(singleton, "SystemVerilog-2009") && FS.find(singleton, "Verilog-2005") === null);
  assert(equal(FS.properties(FS.find(singleton, "SystemVerilog-2023")!), map({
    bodies: ["org.ieee"], number: "1800", revision: ["year", 2023n], names: ["SystemVerilog-2023"] })));
  const editions = new Map<string, string[]>(UNION.branches.map((b: any) => [b.name, [...FS.of(singleton, b.schema).values()].map(
    (s) => FS.tags(s)[FS.tags(s).length - 1] as string)]));
  assert(equal(editions, new Map([...editions.keys()].map((tag) => [tag, SVA.TAGS.includes(tag)
    ? LABELS.slice((SVA.SINCE.get(tag) ?? 3) - 3) : []]))));
  assert(same(LABELS.map((label) => FS.kinds(FS.find(singleton, label)!).length), [14, 17, 17, 17, 17]));
  assert(same([...new Builders.OfObject.Builder(KINDS.get("caseproperty"), false, store).standards].map((e: any) => FS.designationOf(e.standard)),
    IEEE1800.slice(1))); // a kind's schema reads its editions from its side
  assert(store.transaction(SVAssertions.define)[0].get("SystemVerilog_2005") === FS.find(singleton, "SystemVerilog-2005")); // again: nothing new
}

<a id="SVA-07"></a>
## SVA-07 · Printing every kind as mbse-programs writes it: Table 16-3's precedence and associativity, delays and clocks

In [ ]:
// Printing every kind, as mbse-programs' printer writes it: by the precedence of sequence and property operators (Table
// 16-3), right-associative where the standard is, parenthesized only where needed; one that reaches as far right as it
// can (`always`, `if`, a clock, an abort) needs none where nothing follows it. A delay is written after `##` bare where it
// is an integer, a name, a parenthesized expression or a macro usage, else in parentheses.
{
  const clocked = (u: any) => u.clockedsequence((s: any) => s.clock(posedge).sequence(b)); // @(posedge clk) b
  const if_ = (condition: unknown, then: unknown, otherwise: unknown = null) => (u: any) => u.conditionalproperty(
    (s: any) => (otherwise !== null ? s.condition(condition).consequence(then).alternative(otherwise) : s.condition(condition).consequence(then)));
  const abort = (u: any) => u.abortproperty((s: any) => s.keyword("accept_on").condition(rst).operand(a));
  const cases: [unknown, string][] = [
    // sequences
    [delay(a, [one, b], [cycles(one, dollar), c]), "a ##1 b ##[1:$] c"], [delay(null, [two, a]), "##2 a"],
    [delay(a, [n, b]), "a ##n b"], [delay(a, [SV.binary("+", n, one), b]), "a ##(n + 1) b"],
    [delay(a, [(u: any) => u.parenthesizedexpression((p: any) => p.expression(n)), b]), "a ##(n) b"],
    [delay(a, [(u: any) => u.macrousage((m: any) => m.name(spelled("D"))), b]), "a ##`D b"],
    [delay(a, [SV.vector("01"), b]), "a ##2'b01 b"], [delay(a, [SV.literal(1.5), b]), "a ##(1.5) b"],
    [delay(a, [(u: any) => u.constant((k: any) => k.fill("1")), b]), "a ##('1) b"],
    [delay(a, [one, seq(b, "or", c)]), "a ##1 (b or c)"], [delay(seq(a, "or", b), [one, c]), "(a or b) ##1 c"],
    [repeat(a, "*", two), "a[*2]"], [repeat(b, "->", cycles(one, three)), "b[->1:3]"], [repeat(c, "=", n), "c[=n]"],
    [repeat(seq(a, "or", b), "*", two), "(a or b)[*2]"], [repeat(delay(a, [one, b]), "*", two), "(a ##1 b)[*2]"],
    [seq(seq(a, "and", b), "or", c), "a and b or c"], [seq(a, "and", seq(b, "or", c)), "a and (b or c)"],
    [seq(a, "throughout", seq(b, "throughout", c)), "a throughout b throughout c"],
    [seq(seq(a, "throughout", b), "throughout", c), "(a throughout b) throughout c"],
    [seq(a, "within", seq(b, "within", c)), "a within (b within c)"], [seq(a, "intersect", b), "a intersect b"],
    [seq(delay(a, [one, b]), "and", c), "a ##1 b and c"], [seq(SV.binary("&&", a, b), "and", c), "a && b and c"],
    [(u: any) => u.parenthesizedsequence((s: any) => s.sequence(delay(a, [one, b])).items([SV.binary("+", x, one), n])),
      "(a ##1 b, x + 1, n)"],
    [(u: any) => u.firstmatchsequence((s: any) => s.sequence(seq(a, "or", b))), "first_match(a or b)"],
    [clocked, "@(posedge clk) b"], [seq(clocked, "and", c), "(@(posedge clk) b) and c"],
    [seq(a, "and", clocked), "a and @(posedge clk) b"], [delay(a, [one, clocked]), "a ##1 @(posedge clk) b"],
    [delay(a, [one, clocked], [one, c]), "a ##1 (@(posedge clk) b) ##1 c"],
    [(u: any) => u.clockedsequence((s: any) => s.clock((k: any) => k).sequence(a)), "@(*) a"],
    [(u: any) => u.clockedsequence((s: any) => s.clock((k: any) => k.events([(e: any) => e.edge("edge").expression(clk).condition(en),
      (e: any) => e.expression(rst)])).sequence(a)), "@(edge clk iff en or rst) a"],
    // properties
    ...["|->", "|=>", "#-#", "#=#"].map((op) => [implies(a, op, b), `a ${op} b`] as [unknown, string]),
    [implies(a, "|->", implies(b, "|->", c)), "a |-> b |-> c"], [implies(implies(a, "|->", b), "|=>", c), "(a |-> b) |=> c"],
    [implies(delay(a, [one, b]), "|->", seq(c, "or", d)), "a ##1 b |-> c or d"],
    ...["and", "or", "iff", "implies", "until", "s_until", "until_with", "s_until_with"].map(
      (op) => [prop(a, op, b), `a ${op} b`] as [unknown, string]),
    [prop(prop(a, "and", b), "or", c), "a and b or c"], [prop(a, "and", prop(b, "or", c)), "a and (b or c)"],
    [prop(a, "until", prop(b, "until", c)), "a until b until c"], [prop(prop(a, "iff", b), "iff", c), "(a iff b) iff c"],
    [prop(implies(a, "|->", b), "and", c), "(a |-> b) and c"],
    ...["not", "nexttime", "s_nexttime", "always", "s_always", "eventually", "s_eventually"].map(
      (op) => [unary(op, a), `${op} a`] as [unknown, string]),
    [unary("nexttime", a, two), "nexttime [2] a"], [unary("always", a, cycles(one, three)), "always [1:3] a"],
    [unary("not", prop(a, "and", b)), "not (a and b)"], [prop(unary("not", a), "and", b), "not a and b"],
    [unary("always", prop(a, "and", b)), "always a and b"], [prop(unary("always", a), "and", b), "(always a) and b"],
    [prop(a, "and", unary("always", b)), "a and always b"], [implies(a, "|->", unary("s_eventually", b)), "a |-> s_eventually b"],
    [(u: any) => u.strengthproperty((s: any) => s.keyword("strong").sequence(seq(a, "or", b))), "strong(a or b)"],
    [(u: any) => u.strengthproperty((s: any) => s.keyword("weak").sequence(a)), "weak(a)"],
    [abort, "accept_on (rst) a"], [prop(abort, "or", b), "(accept_on (rst) a) or b"], [prop(b, "or", abort), "b or accept_on (rst) a"],
    ...["reject_on", "sync_accept_on", "sync_reject_on"].map((k) => [(u: any) => u.abortproperty(
      (s: any) => s.keyword(k).condition(rst).operand(a)), `${k} (rst) a`] as [unknown, string]),
    [if_(en, a), "if (en) a"], [if_(en, a, b), "if (en) a else b"], [if_(en, if_(rst, a), b), "if (en) (if (rst) a) else b"],
    [if_(en, a, if_(rst, b, c)), "if (en) a else if (rst) b else c"], [prop(if_(en, a), "and", b), "(if (en) a) and b"],
    [(u: any) => u.caseproperty((s: any) => s.expression(n).items([(t: any) => t.expressions([one, two]).body(a),
      (t: any) => t.body(implies(b, "|->", c))])), "case (n) 1, 2: a; default: b |-> c; endcase"],
    [(u: any) => u.parenthesizedproperty((s: any) => s.property(implies(a, "|->", b))), "(a |-> b)"],
    [(u: any) => u.clockedproperty((s: any) => s.clock(posedge).property(implies(a, "|->", b))), "@(posedge clk) a |-> b"],
    [prop((u: any) => u.clockedproperty((s: any) => s.clock(posedge).property(a)), "and", b), "(@(posedge clk) a) and b"],
    // a property spec and a cycle range
    [(u: any) => u.propertyspec((s: any) => s.clock(posedge).disable(rst).property(implies(a, "|=>", b))),
      "@(posedge clk) disable iff (rst) a |=> b"],
    [(u: any) => u.propertyspec((s: any) => s.disable(rst).property(a)), "disable iff (rst) a"],
    [(u: any) => u.propertyspec((s: any) => s.property(a)), "a"], [cycles(one, dollar), "1:$"],
  ];
  for (const [spec, written] of cases) assert(text(spec) === written, `${written} ${text(spec)}`);
}

<a id="SVA-08"></a>
## SVA-08 · Evaluation refuses each of its kinds by name, as a root and as a call's argument

In [ ]:
// Evaluation (Q11): SystemVerilog's evaluator refuses each of its kinds by name, as a root and where a call's argument
// holds one; what it evaluated before, an expression among them, it still evaluates.
{
  const SAMPLES = new Map<string, unknown>([
    ["delaysequence", delay(a, [one, b])], ["repetitionsequence", repeat(a, "*", two)], ["binarysequence", seq(a, "and", b)],
    ["parenthesizedsequence", (u: any) => u.parenthesizedsequence((s: any) => s.sequence(a))],
    ["firstmatchsequence", (u: any) => u.firstmatchsequence((s: any) => s.sequence(a))],
    ["clockedsequence", (u: any) => u.clockedsequence((s: any) => s.clock(posedge).sequence(a))],
    ["implicationproperty", implies(a, "|->", b)], ["binaryproperty", prop(a, "and", b)], ["unaryproperty", unary("not", a)],
    ["strengthproperty", (u: any) => u.strengthproperty((s: any) => s.keyword("weak").sequence(a))],
    ["abortproperty", (u: any) => u.abortproperty((s: any) => s.keyword("accept_on").condition(rst).operand(a))],
    ["conditionalproperty", (u: any) => u.conditionalproperty((s: any) => s.condition(en).consequence(a))],
    ["caseproperty", (u: any) => u.caseproperty((s: any) => s.expression(n).items([(t: any) => t.body(a)]))],
    ["parenthesizedproperty", (u: any) => u.parenthesizedproperty((s: any) => s.property(a))],
    ["clockedproperty", (u: any) => u.clockedproperty((s: any) => s.clock(posedge).property(a))],
    ["propertyspec", (u: any) => u.propertyspec((s: any) => s.property(a))], ["cyclerange", cycles(one, two)],
  ]);
  assert(same([...SAMPLES.keys()], SVA.TAGS));
  const values = { a: 1n, b: 0n, c: 1n, d: 0n, clk: 1n, rst: 0n, en: 1n, n: 2n, x: 3n };
  for (const [tag, spec] of SAMPLES) {
    raises(NotImplementedError, () => SVV.OfAny(held(spec), values), `SystemVerilog Assertions ${tag} is not evaluated`);
  }
  raises(NotImplementedError, () => SVV.OfAny(held(SV.call("f", SAMPLES.get("delaysequence")), "Expressions"),
    new SVV.Scope(values, { functions: { f: (value: unknown) => value } })), "SystemVerilog Assertions delaysequence is not evaluated");
  assert(String(SVV.OfAny(held(SV.binary("+", a, n)), values)) === "32'sb00000000000000000000000000000011" && String(SVV.OfAny(held(SV.binary("+", a, n), "Expressions"), values)) === "32'sb00000000000000000000000000000011");
}